# TDA Failure Analysis and Selective Admission

This notebook reproduces the experiments conducted for the Visual Media
course report.

## Experiments

1. Environment and dataset setup
2. Known-feature precomputation
3. Failure Case 1: Adverse Test Ordering
4. Unknown-feature precomputation
5. Failure Case 2: Unknown-Class Contamination
6. Improvement: Cache-Specific Selective Admission

The experiments use Caltech101, CLIP-ResNet-50, and the official TDA
configuration.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import torch


# Set True to recompute features and rerun all experiments.
FORCE_RERUN = False

REPOSITORY_URL = (
    "https://github.com/koyar4271/TDA-VisualMedia.git"
)
REPOSITORY_ROOT = Path("/content/TDA")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/VisualMedia/TDA"
)
ASSET_ROOT = DRIVE_ROOT / "assets"
FEATURE_ROOT = DRIVE_ROOT / "features"

CALTECH_ARCHIVE = ASSET_ROOT / "caltech-101.zip"
CALTECH_SPLIT_CACHE = (
    ASSET_ROOT / "split_zhou_Caltech101.json"
)

LOCAL_DATASET_ROOT = (
    REPOSITORY_ROOT / "dataset" / "caltech-101"
)
LOCAL_IMAGE_ROOT = (
    LOCAL_DATASET_ROOT / "101_ObjectCategories"
)
LOCAL_UNKNOWN_ROOT = (
    LOCAL_IMAGE_ROOT / "BACKGROUND_Google"
)
LOCAL_SPLIT_FILE = (
    LOCAL_DATASET_ROOT / "split_zhou_Caltech101.json"
)

KNOWN_FEATURE_FILE = (
    FEATURE_ROOT / "caltech101_rn50_features.pt"
)
UNKNOWN_FEATURE_FILE = (
    FEATURE_ROOT
    / "caltech101_background_rn50_features.pt"
)

ADVERSE_OUTPUT_ROOT = (
    DRIVE_ROOT / "adverse_ordering_results"
)
UNKNOWN_OUTPUT_ROOT = (
    DRIVE_ROOT / "unknown_contamination_results"
)
SELECTIVE_OUTPUT_ROOT = (
    DRIVE_ROOT / "selective_admission_results"
)

CONFIG_FILE = (
    REPOSITORY_ROOT / "configs" / "caltech101.yaml"
)


def run_checked(
    command: list[str],
    cwd: Path = REPOSITORY_ROOT,
) -> None:
    """Run a command and raise an exception if it fails."""
    command = [str(value) for value in command]
    print("$", " ".join(command))
    subprocess.run(
        command,
        cwd=cwd,
        check=True,
    )


def prepare_output_directory(
    output_root: Path,
) -> None:
    """Create an output directory or reset it when requested."""
    if FORCE_RERUN and output_root.exists():
        shutil.rmtree(output_root)

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )


def load_torch_artifact(path: Path):
    """Load a torch artifact across PyTorch versions."""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )

Mounted at /content/drive


In [2]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. "
        "Select a GPU runtime in Google Colab."
    )

print("GPU:", torch.cuda.get_device_name(0))

# Clone a clean copy of the repository.
os.chdir("/content")

if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)

run_checked(
    [
        "git",
        "clone",
        REPOSITORY_URL,
        str(REPOSITORY_ROOT),
    ],
    cwd=Path("/content"),
)

# Install dependencies.
requirements_file = (
    REPOSITORY_ROOT / "requirements-colab.txt"
)

if requirements_file.is_file():
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_file),
    ]
else:
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyyaml",
        "yacs",
        "gdown",
        "ftfy",
        "regex",
        "tqdm",
        "wandb>=0.22.3",
        "chardet",
        "future",
        "scipy",
        "scikit-learn",
        "tabulate",
        "pandas",
        "matplotlib",
    ]

run_checked(
    install_command,
    cwd=REPOSITORY_ROOT,
)

ASSET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
FEATURE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Persist downloaded CLIP weights across Colab sessions.
drive_clip_cache = ASSET_ROOT / "clip"
drive_clip_cache.mkdir(
    parents=True,
    exist_ok=True,
)

local_clip_cache = Path.home() / ".cache" / "clip"
local_clip_cache.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if local_clip_cache.is_symlink():
    local_clip_cache.unlink()
elif local_clip_cache.exists():
    if local_clip_cache.is_file():
        local_clip_cache.unlink()
    else:
        shutil.rmtree(local_clip_cache)

local_clip_cache.symlink_to(
    drive_clip_cache,
    target_is_directory=True,
)

# Download and cache the Caltech101 archive.
if not CALTECH_ARCHIVE.is_file():
    archive_url = (
        "https://data.caltech.edu/records/"
        "mzrjq-6wc02/files/"
        "caltech-101.zip?download=1"
    )
    temporary_archive = (
        CALTECH_ARCHIVE.with_suffix(".zip.part")
    )

    if temporary_archive.exists():
        temporary_archive.unlink()

    print("Downloading the Caltech101 archive...")

    run_checked(
        [
            "wget",
            "--quiet",
            "--show-progress",
            "--user-agent=Mozilla/5.0",
            "--output-document",
            str(temporary_archive),
            archive_url,
        ],
        cwd=Path("/content"),
    )

    if not temporary_archive.is_file():
        raise FileNotFoundError(
            "The downloaded archive was not found."
        )

    if temporary_archive.stat().st_size < 100_000_000:
        raise RuntimeError(
            "The downloaded archive is unexpectedly small."
        )

    temporary_archive.replace(CALTECH_ARCHIVE)
else:
    print("Using the cached Caltech101 archive.")

# Extract the dataset into the current Colab runtime.
if not LOCAL_IMAGE_ROOT.is_dir():
    temporary_root = Path("/content/caltech_setup")
    outer_extract_root = temporary_root / "outer"
    inner_extract_root = temporary_root / "inner"

    if temporary_root.exists():
        shutil.rmtree(temporary_root)

    outer_extract_root.mkdir(parents=True)
    inner_extract_root.mkdir(parents=True)

    print("Extracting the Caltech101 archive...")

    with zipfile.ZipFile(CALTECH_ARCHIVE) as archive:
        archive.extractall(outer_extract_root)

    inner_archives = list(
        outer_extract_root.rglob(
            "101_ObjectCategories.tar.gz"
        )
    )

    if not inner_archives:
        raise FileNotFoundError(
            "101_ObjectCategories.tar.gz was not found."
        )

    with tarfile.open(
        inner_archives[0],
        "r:gz",
    ) as archive:
        try:
            archive.extractall(
                inner_extract_root,
                filter="data",
            )
        except TypeError:
            archive.extractall(inner_extract_root)

    source_directories = [
        path
        for path in inner_extract_root.rglob(
            "101_ObjectCategories"
        )
        if path.is_dir()
    ]

    if not source_directories:
        raise FileNotFoundError(
            "101_ObjectCategories was not found "
            "after extraction."
        )

    LOCAL_DATASET_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.move(
        str(source_directories[0]),
        str(LOCAL_IMAGE_ROOT),
    )
    shutil.rmtree(temporary_root)
else:
    print(
        "The local Caltech101 image directory "
        "already exists."
    )

# Download and cache the official dataset split.
if not CALTECH_SPLIT_CACHE.is_file():
    import gdown

    temporary_split = (
        CALTECH_SPLIT_CACHE.with_suffix(".json.part")
    )

    if temporary_split.exists():
        temporary_split.unlink()

    print("Downloading the Caltech101 split file...")

    download_result = gdown.download(
        id="1hyarUivQE36mY6jSomru6Fjd-JzwcCzN",
        output=str(temporary_split),
        quiet=False,
    )

    if not download_result:
        raise RuntimeError(
            "The Caltech101 split file download failed."
        )

    temporary_split.replace(CALTECH_SPLIT_CACHE)
else:
    print("Using the cached Caltech101 split file.")

LOCAL_DATASET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    CALTECH_SPLIT_CACHE,
    LOCAL_SPLIT_FILE,
)

# Verify all files required by the notebook.
experiment_scripts = [
    "precompute_features.py",
    "run_adverse_ordering.py",
    "precompute_unknown_features.py",
    "run_unknown_contamination.py",
    "run_selective_admission.py",
]

required_paths = [
    REPOSITORY_ROOT / "clip" / "__init__.py",
    CONFIG_FILE,
    LOCAL_IMAGE_ROOT,
    LOCAL_UNKNOWN_ROOT,
    LOCAL_SPLIT_FILE,
]

required_paths.extend(
    REPOSITORY_ROOT / "experiments" / file_name
    for file_name in experiment_scripts
)

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    formatted_paths = "\n".join(
        str(path)
        for path in missing_paths
    )
    raise FileNotFoundError(
        "Required project files are missing:\n"
        f"{formatted_paths}"
    )

# Check the syntax of all experiment scripts.
run_checked(
    [
        sys.executable,
        "-m",
        "compileall",
        "-q",
        "experiments",
    ],
    cwd=REPOSITORY_ROOT,
)

print("Repository:", REPOSITORY_ROOT)
print("Dataset:", LOCAL_DATASET_ROOT)
print("The experiment environment is ready.")

GPU: Tesla T4
$ git clone https://github.com/koyar4271/TDA-VisualMedia.git /content/TDA
$ /usr/bin/python3 -m pip install -q pyyaml yacs gdown ftfy regex tqdm wandb>=0.22.3 chardet future scipy scikit-learn tabulate pandas matplotlib
Using the cached Caltech101 archive.
Extracting the Caltech101 archive...
Using the cached Caltech101 split file.
$ /usr/bin/python3 -m compileall -q experiments
Repository: /content/TDA
Dataset: /content/TDA/dataset/caltech-101
The experiment environment is ready.


## Known-Feature Precomputation

Precompute CLIP image features and zero-shot predictions for the
Caltech101 known-class test set.

The resulting feature file is cached in Google Drive and reused across
experiments.

In [3]:
if FORCE_RERUN and KNOWN_FEATURE_FILE.exists():
    KNOWN_FEATURE_FILE.unlink()

KNOWN_FEATURE_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if KNOWN_FEATURE_FILE.is_file():
    print(
        "Using the cached known feature file:",
        KNOWN_FEATURE_FILE,
    )
else:
    run_checked(
        [
            sys.executable,
            "experiments/precompute_features.py",
            "--dataset",
            "caltech101",
            "--data-root",
            "./dataset",
            "--backbone",
            "RN50",
            "--batch-size",
            "1",
            "--num-workers",
            "2",
            "--output",
            str(KNOWN_FEATURE_FILE),
        ]
    )

if not KNOWN_FEATURE_FILE.is_file():
    raise FileNotFoundError(
        "Known-feature precomputation did not "
        "produce the expected file."
    )

known_artifact = load_torch_artifact(
    KNOWN_FEATURE_FILE
)

clip_accuracy = (
    known_artifact["clip_predictions"]
    == known_artifact["labels"]
).float().mean().item() * 100.0

print("Known feature file:", KNOWN_FEATURE_FILE)
print(f"CLIP accuracy: {clip_accuracy:.3f}%")

Using the cached known feature file: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_rn50_features.pt
Known feature file: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_rn50_features.pt
CLIP accuracy: 87.830%


## Failure Case 1: Adverse Test Ordering

This experiment compares the following test-stream orderings:

- Random ordering with five seeds
- Class-balanced easy-first ordering
- Class-balanced uncertain-first ordering
- Confidently-wrong-first ordering

In [4]:
ADVERSE_SUMMARY_FILE = (
    ADVERSE_OUTPUT_ROOT
    / "summaries"
    / "adverse_ordering_summary.csv"
)

prepare_output_directory(
    ADVERSE_OUTPUT_ROOT
)

if ADVERSE_SUMMARY_FILE.is_file() and not FORCE_RERUN:
    print(
        "Using existing adverse-ordering results:",
        ADVERSE_SUMMARY_FILE,
    )
else:
    run_checked(
        [
            sys.executable,
            "experiments/run_adverse_ordering.py",
            "--features",
            str(KNOWN_FEATURE_FILE),
            "--config",
            str(CONFIG_FILE),
            "--output-root",
            str(ADVERSE_OUTPUT_ROOT),
            "--interval-size",
            "100",
            "--conditions",
            "random",
            "class_balanced_easy_first",
            "class_balanced_uncertain_first",
            "confidently_wrong_first",
            "--random-seeds",
            "0",
            "1",
            "2",
            "3",
            "4",
            "--deterministic-seed",
            "0",
            "--stress-prefix-size",
            "100",
            "--device",
            "cuda",
        ]
    )

if not ADVERSE_SUMMARY_FILE.is_file():
    raise FileNotFoundError(
        "The adverse-ordering summary was not generated."
    )

adverse_summary_df = pd.read_csv(
    ADVERSE_SUMMARY_FILE
)

display(adverse_summary_df)

print(
    "Generated figures:",
    ADVERSE_OUTPUT_ROOT / "figures",
)

Using existing adverse-ordering results: /content/drive/MyDrive/VisualMedia/TDA/adverse_ordering_results/summaries/adverse_ordering_summary.csv


,run_name,condition,seed,num_samples,stress_prefix_size,clip_accuracy,tda_accuracy,tda_minus_clip_pp,clip_correct_tda_wrong_count,positive_admission_count,positive_admission_purity,final_positive_cache_purity,mean_positive_cache_purity,wrong_cache_entry_count,wrong_cache_lifetime_mean,wrong_cache_lifetime_median,wrong_cache_lifetime_max,wrong_cache_survival_to_end_rate
0,random_seed_0,random,0,2465,0,87.829615,89.533469,1.703854,19,837,0.917563,0.98,0.941470,69,489.333333,412.0,2411,0.086957
1,random_seed_1,random,1,2465,0,87.829615,89.452333,1.622718,27,812,0.922414,0.98,0.943438,63,486.984127,317.0,1805,0.095238
2,random_seed_2,random,2,2465,0,87.829615,89.492901,1.663286,26,812,0.927340,0.98,0.944299,59,541.033898,349.0,2335,0.101695
3,random_seed_3,random,3,2465,0,87.829615,89.695740,1.866126,21,801,0.920100,0.98,0.934224,64,581.937500,350.5,2457,0.093750
4,random_seed_4,random,4,2465,0,87.829615,89.452333,1.622718,21,821,0.923264,0.98,0.939441,63,509.857143,344.0,2357,0.095238
5,class_balanced_easy_first,class_balanced_easy_first,0,2465,0,87.829615,89.371197,1.541582,29,315,0.946032,0.98,0.981278,17,613.941176,199.0,2406,0.352941
6,class_balanced_uncertain_first,class_balanced_uncertain_first,0,2465,0,87.829615,89.736308,1.906694,2,2327,0.926085,0.98,0.895497,172,317.005814,232.0,2103,0.034884
7,confidently_wrong_first,confidently_wrong_first,0,2465,100,87.829615,88.884381,1.054767,27,866,0.857968,0.98,0.827909,123,643.983740,473.0,2463,0.048780


Generated figures: /content/drive/MyDrive/VisualMedia/TDA/adverse_ordering_results/figures


## Unknown-Feature Precomputation

Precompute CLIP features and known-class predictions for the
`BACKGROUND_Google` images.

These images are treated as the unknown-image source in Failure Case 2.

In [5]:
if FORCE_RERUN and UNKNOWN_FEATURE_FILE.exists():
    UNKNOWN_FEATURE_FILE.unlink()

UNKNOWN_FEATURE_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if UNKNOWN_FEATURE_FILE.is_file():
    print(
        "Using the cached unknown feature file:",
        UNKNOWN_FEATURE_FILE,
    )
else:
    run_checked(
        [
            sys.executable,
            "experiments/precompute_unknown_features.py",
            "--known-features",
            str(KNOWN_FEATURE_FILE),
            "--data-root",
            "./dataset",
            "--unknown-directory",
            str(LOCAL_UNKNOWN_ROOT),
            "--batch-size",
            "32",
            "--num-workers",
            "2",
            "--output",
            str(UNKNOWN_FEATURE_FILE),
        ]
    )

if not UNKNOWN_FEATURE_FILE.is_file():
    raise FileNotFoundError(
        "Unknown-feature precomputation did not "
        "produce the expected file."
    )

unknown_artifact = load_torch_artifact(
    UNKNOWN_FEATURE_FILE
)

print("Known samples:", len(known_artifact["labels"]))
print("Unknown samples:", len(unknown_artifact["features"]))
print("Unknown feature file:", UNKNOWN_FEATURE_FILE)

Using the cached unknown feature file: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_background_rn50_features.pt
Known samples: 2465
Unknown samples: 467
Unknown feature file: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_background_rn50_features.pt


## Failure Case 2: Unknown-Class Contamination

The test stream has the following structure:

`Known prefix -> Unknown block -> Known suffix`

The experiment compares the contaminated known suffix with the same
known suffix processed without unknown images.

In [6]:
UNKNOWN_SUMMARY_FILE = (
    UNKNOWN_OUTPUT_ROOT
    / "summaries"
    / "unknown_contamination_summary.csv"
)

prepare_output_directory(
    UNKNOWN_OUTPUT_ROOT
)

if UNKNOWN_SUMMARY_FILE.is_file() and not FORCE_RERUN:
    print(
        "Using existing unknown-contamination results:",
        UNKNOWN_SUMMARY_FILE,
    )
else:
    run_checked(
        [
            sys.executable,
            "experiments/run_unknown_contamination.py",
            "--known-features",
            str(KNOWN_FEATURE_FILE),
            "--unknown-features",
            str(UNKNOWN_FEATURE_FILE),
            "--config",
            str(CONFIG_FILE),
            "--output-root",
            str(UNKNOWN_OUTPUT_ROOT),
            "--known-order-seed",
            "0",
            "--prefix-sizes",
            "100",
            "500",
            "--unknown-counts",
            "0",
            "25",
            "50",
            "100",
            "200",
            "--unknown-seeds",
            "0",
            "1",
            "2",
            "3",
            "4",
            "--recovery-bin-size",
            "50",
            "--main-prefix-size",
            "100",
            "--device",
            "cuda",
        ]
    )

if not UNKNOWN_SUMMARY_FILE.is_file():
    raise FileNotFoundError(
        "The unknown-contamination summary "
        "was not generated."
    )

print(
    "Experiment results:",
    UNKNOWN_OUTPUT_ROOT,
)

Using existing unknown-contamination results: /content/drive/MyDrive/VisualMedia/TDA/unknown_contamination_results/summaries/unknown_contamination_summary.csv
Experiment results: /content/drive/MyDrive/VisualMedia/TDA/unknown_contamination_results


In [7]:
unknown_summary_df = pd.read_csv(
    UNKNOWN_SUMMARY_FILE
)

selected_unknown_columns = [
    "run_name",
    "prefix_size",
    "unknown_count",
    "unknown_seed",
    "control_suffix_accuracy",
    "contaminated_suffix_accuracy",
    "paired_accuracy_difference_pp",
    "unknown_positive_admission_rate",
    "unknown_negative_admission_rate",
    "peak_positive_contamination_ratio",
    "peak_negative_contamination_ratio",
    "unknown_harmful_flip_count",
]

available_unknown_columns = [
    column
    for column in selected_unknown_columns
    if column in unknown_summary_df.columns
]

display(
    unknown_summary_df[
        available_unknown_columns
    ]
)

unknown_aggregated_df = (
    unknown_summary_df[
        unknown_summary_df["unknown_count"] > 0
    ]
    .groupby(
        [
            "prefix_size",
            "unknown_count",
        ],
        as_index=False,
    )
    .agg(
        paired_accuracy_mean=(
            "paired_accuracy_difference_pp",
            "mean",
        ),
        paired_accuracy_std=(
            "paired_accuracy_difference_pp",
            "std",
        ),
        contaminated_accuracy_mean=(
            "contaminated_suffix_accuracy",
            "mean",
        ),
        positive_admission_rate_mean=(
            "unknown_positive_admission_rate",
            "mean",
        ),
        negative_admission_rate_mean=(
            "unknown_negative_admission_rate",
            "mean",
        ),
        peak_positive_contamination_mean=(
            "peak_positive_contamination_ratio",
            "mean",
        ),
        peak_negative_contamination_mean=(
            "peak_negative_contamination_ratio",
            "mean",
        ),
        harmful_flip_mean=(
            "unknown_harmful_flip_count",
            "mean",
        ),
    )
    .fillna(0.0)
)

display(unknown_aggregated_df)

,run_name,prefix_size,unknown_count,unknown_seed,control_suffix_accuracy,contaminated_suffix_accuracy,paired_accuracy_difference_pp,unknown_positive_admission_rate,unknown_negative_admission_rate,peak_positive_contamination_ratio,peak_negative_contamination_ratio,unknown_harmful_flip_count
0,prefix_100_unknown_0_control,100,0,-1,89.640592,89.640592,0.000000,0.000,0.000,0.000000,0.000000,0
1,prefix_100_unknown_25_seed_0,100,25,0,89.640592,89.640592,0.000000,0.960,0.520,0.223301,0.464286,2
2,prefix_100_unknown_25_seed_1,100,25,1,89.640592,89.682875,0.042283,0.960,0.280,0.211538,0.318182,0
3,prefix_100_unknown_25_seed_2,100,25,2,89.640592,89.598309,-0.042283,0.840,0.760,0.196078,0.545455,2
4,prefix_100_unknown_25_seed_3,100,25,3,89.640592,89.682875,0.042283,0.800,0.560,0.188119,0.464286,0
5,prefix_100_unknown_25_seed_4,100,25,4,89.640592,89.260042,-0.380550,0.840,0.520,0.205882,0.464286,1
6,prefix_100_unknown_50_seed_0,100,50,0,89.640592,89.725159,0.084567,0.900,0.580,0.349593,0.651163,3
7,prefix_100_unknown_50_seed_1,100,50,1,89.640592,89.682875,0.042283,0.860,0.440,0.327731,0.594595,3
8,prefix_100_unknown_50_seed_2,100,50,2,89.640592,89.344609,-0.295983,0.780,0.620,0.301724,0.659091,3
9,prefix_100_unknown_50_seed_3,100,50,3,89.640592,89.640592,0.000000,0.820,0.600,0.316667,0.659091,1


,prefix_size,unknown_count,paired_accuracy_mean,paired_accuracy_std,contaminated_accuracy_mean,positive_admission_rate_mean,negative_admission_rate_mean,peak_positive_contamination_mean,peak_negative_contamination_mean,harmful_flip_mean
0,100,25,-0.067653,0.178393,89.572939,0.880,0.528,0.204984,0.451299,1.0
1,100,50,-0.109937,0.212681,89.530655,0.848,0.552,0.326356,0.639617,2.6
2,100,100,-0.084567,0.174339,89.556025,0.794,0.540,0.467152,0.773429,2.6
3,100,200,-0.228330,0.168073,89.412262,0.712,0.514,0.598828,0.864851,5.0
4,500,25,-0.050891,0.176290,89.872774,0.328,0.528,0.029852,0.206727,0.6
5,500,50,-0.081425,0.211671,89.842239,0.276,0.532,0.048608,0.344259,2.0
6,500,100,-0.040712,0.166467,89.882952,0.244,0.506,0.079955,0.494519,1.6
7,500,200,-0.183206,0.159313,89.740458,0.220,0.474,0.129383,0.635241,3.6


## Improvement: Cache-Specific Selective Admission

The admission gate uses:

- The normalized prediction entropy
- The maximum image-text similarity

Thresholds are calibrated without labels from the first 100 known
samples.

In [8]:
SELECTIVE_COMPARISON_FILE = (
    SELECTIVE_OUTPUT_ROOT
    / "summaries"
    / "selective_admission_comparison.csv"
)

SELECTIVE_CLEAN_FILE = (
    SELECTIVE_OUTPUT_ROOT
    / "summaries"
    / "selective_admission_clean_random.csv"
)

prepare_output_directory(
    SELECTIVE_OUTPUT_ROOT
)

if (
    SELECTIVE_COMPARISON_FILE.is_file()
    and SELECTIVE_CLEAN_FILE.is_file()
    and not FORCE_RERUN
):
    print(
        "Using existing selective-admission results:",
        SELECTIVE_OUTPUT_ROOT,
    )
else:
    run_checked(
        [
            sys.executable,
            "experiments/run_selective_admission.py",
            "--known-features",
            str(KNOWN_FEATURE_FILE),
            "--unknown-features",
            str(UNKNOWN_FEATURE_FILE),
            "--config",
            str(CONFIG_FILE),
            "--output-root",
            str(SELECTIVE_OUTPUT_ROOT),
            "--clean-known-order-seeds",
            "0",
            "1",
            "2",
            "3",
            "4",
            "--contamination-known-order-seed",
            "0",
            "--prefix-sizes",
            "100",
            "500",
            "--unknown-counts",
            "25",
            "50",
            "100",
            "200",
            "--unknown-seeds",
            "0",
            "1",
            "2",
            "3",
            "4",
            "--calibration-size",
            "100",
            "--pos-entropy-quantile",
            "0.90",
            "--similarity-quantile",
            "0.05",
            "--recovery-bin-size",
            "50",
            "--main-prefix-size",
            "100",
            "--device",
            "cuda",
        ]
    )

for required_result in [
    SELECTIVE_COMPARISON_FILE,
    SELECTIVE_CLEAN_FILE,
]:
    if not required_result.is_file():
        raise FileNotFoundError(
            f"Expected result was not generated: "
            f"{required_result}"
        )

print(
    "Experiment results:",
    SELECTIVE_OUTPUT_ROOT,
)

Using existing selective-admission results: /content/drive/MyDrive/VisualMedia/TDA/selective_admission_results
Experiment results: /content/drive/MyDrive/VisualMedia/TDA/selective_admission_results


In [9]:
comparison_df = pd.read_csv(
    SELECTIVE_COMPARISON_FILE
)

comparison_aggregated_df = (
    comparison_df.groupby(
        [
            "prefix_size",
            "unknown_count",
        ],
        as_index=False,
    )
    .agg(
        original_effect_mean=(
            "original_contamination_effect_pp",
            "mean",
        ),
        original_effect_std=(
            "original_contamination_effect_pp",
            "std",
        ),
        selective_effect_mean=(
            "selective_contamination_effect_pp",
            "mean",
        ),
        selective_effect_std=(
            "selective_contamination_effect_pp",
            "std",
        ),
        contaminated_accuracy_gain_mean=(
            "contaminated_accuracy_gain_pp",
            "mean",
        ),
        mitigation_mean=(
            "mitigation_pp",
            "mean",
        ),
        original_positive_contamination_mean=(
            "original_peak_positive_contamination_ratio",
            "mean",
        ),
        selective_positive_contamination_mean=(
            "selective_peak_positive_contamination_ratio",
            "mean",
        ),
        original_negative_contamination_mean=(
            "original_peak_negative_contamination_ratio",
            "mean",
        ),
        selective_negative_contamination_mean=(
            "selective_peak_negative_contamination_ratio",
            "mean",
        ),
        original_harmful_flip_mean=(
            "original_harmful_flip_count",
            "mean",
        ),
        selective_harmful_flip_mean=(
            "selective_harmful_flip_count",
            "mean",
        ),
    )
    .fillna(0.0)
)

display(comparison_aggregated_df)

clean_df = pd.read_csv(
    SELECTIVE_CLEAN_FILE
)

clean_summary_df = (
    clean_df.groupby(
        "method",
        as_index=False,
    )
    .agg(
        full_accuracy_mean=(
            "full_accuracy",
            "mean",
        ),
        full_accuracy_std=(
            "full_accuracy",
            "std",
        ),
        suffix_accuracy_mean=(
            "suffix_accuracy",
            "mean",
        ),
        suffix_accuracy_std=(
            "suffix_accuracy",
            "std",
        ),
        known_positive_admission_mean=(
            "known_positive_admission_rate",
            "mean",
        ),
        known_negative_admission_mean=(
            "known_negative_admission_rate",
            "mean",
        ),
    )
    .fillna(0.0)
)

display(clean_summary_df)

,prefix_size,unknown_count,original_effect_mean,original_effect_std,selective_effect_mean,selective_effect_std,contaminated_accuracy_gain_mean,mitigation_mean,original_positive_contamination_mean,selective_positive_contamination_mean,original_negative_contamination_mean,selective_negative_contamination_mean,original_harmful_flip_mean,selective_harmful_flip_mean
0,100,25,-0.067653,0.178393,-0.025370,0.048210,-4.228330e-02,0.042283,0.204984,0.030700,0.451299,0.151664,1.0,0.6
1,100,50,-0.109937,0.212681,-0.076110,0.055131,-5.073996e-02,0.033827,0.326356,0.065609,0.639617,0.303981,2.6,1.8
2,100,100,-0.084567,0.174339,-0.016913,0.048210,-1.691332e-02,0.067653,0.467152,0.127539,0.773429,0.464972,2.6,1.2
3,100,200,-0.228330,0.168073,-0.050740,0.081334,9.302326e-02,0.177590,0.598828,0.202149,0.864851,0.626508,5.0,1.6
4,500,25,-0.050891,0.176290,0.000000,0.035985,0.000000e+00,0.050891,0.029852,0.007984,0.206727,0.056101,0.6,0.2
5,500,50,-0.081425,0.211671,-0.030534,0.077179,5.706546e-15,0.050891,0.048608,0.013511,0.344259,0.123789,2.0,0.8
6,500,100,-0.040712,0.166467,0.040712,0.055748,3.053435e-02,0.081425,0.079955,0.025242,0.494519,0.214968,1.6,0.4
7,500,200,-0.183206,0.159313,0.000000,0.080465,1.323155e-01,0.183206,0.129383,0.037650,0.635241,0.347160,3.6,1.0


,method,full_accuracy_mean,full_accuracy_std,suffix_accuracy_mean,suffix_accuracy_std,known_positive_admission_mean,known_negative_admission_mean
0,original,89.525355,0.101013,89.606765,0.109448,0.30723,0.078901
1,selective,89.371197,0.137573,89.446089,0.157075,0.29556,0.073827


In [10]:
result_files = {
    "Known features": KNOWN_FEATURE_FILE,
    "Unknown features": UNKNOWN_FEATURE_FILE,
    "Adverse ordering summary": ADVERSE_SUMMARY_FILE,
    "Unknown contamination summary": UNKNOWN_SUMMARY_FILE,
    "Selective comparison": SELECTIVE_COMPARISON_FILE,
    "Selective clean results": SELECTIVE_CLEAN_FILE,
}

missing_results = []

for name, path in result_files.items():
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{status}] {name}: {path}")

    if not path.is_file():
        missing_results.append(path)

if missing_results:
    raise FileNotFoundError(
        "Some expected experiment artifacts are missing."
    )

figure_directories = [
    ADVERSE_OUTPUT_ROOT / "figures",
    UNKNOWN_OUTPUT_ROOT / "figures",
    SELECTIVE_OUTPUT_ROOT / "figures",
]

for figure_directory in figure_directories:
    print(f"\nFigures in {figure_directory}:")

    if not figure_directory.is_dir():
        print("  Directory not found.")
        continue

    for figure_path in sorted(
        figure_directory.glob("*.png")
    ):
        print(" ", figure_path.name)

print("\nAll expected experiment artifacts are available.")

[OK] Known features: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_rn50_features.pt
[OK] Unknown features: /content/drive/MyDrive/VisualMedia/TDA/features/caltech101_background_rn50_features.pt
[OK] Adverse ordering summary: /content/drive/MyDrive/VisualMedia/TDA/adverse_ordering_results/summaries/adverse_ordering_summary.csv
[OK] Unknown contamination summary: /content/drive/MyDrive/VisualMedia/TDA/unknown_contamination_results/summaries/unknown_contamination_summary.csv
[OK] Selective comparison: /content/drive/MyDrive/VisualMedia/TDA/selective_admission_results/summaries/selective_admission_comparison.csv
[OK] Selective clean results: /content/drive/MyDrive/VisualMedia/TDA/selective_admission_results/summaries/selective_admission_clean_random.csv

Figures in /content/drive/MyDrive/VisualMedia/TDA/adverse_ordering_results/figures:
  adverse_ordering_cumulative_gap.png
  adverse_ordering_final_accuracy_gap.png
  adverse_ordering_interval_accuracy.png
  adverse_ordering_po